In [4]:
import os
import re
import warnings
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import spacy
from sklearn.linear_model import LogisticRegression
import numpy as np
from gensim.models import Word2Vec, FastText
import gensim.downloader as api
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

In [5]:
def load_corpus(path: str) -> list[str]:
    """
    Accepts .txt, .csv, or .xlsx.
    Returns a list of non-empty sentence strings.
    """
    ext = os.path.splitext(path)[-1].lower()

    if ext == ".txt":
        with open(path, encoding="utf-8", errors="ignore") as f:
            raw = f.read()
        # split on sentence-ending punctuation
        sentences = re.split(r'(?<=[.!?])\s+', raw)

    elif ext == ".csv":
        df = pd.read_csv(path, encoding="utf-8", encoding_errors="replace")
        # try common column names
        col = next((c for c in df.columns
                    if "sentence" in c.lower() or "text" in c.lower()
                    or "token" in c.lower()), df.columns[0])
        sentences = df[col].dropna().astype(str).tolist()

    elif ext in (".xlsx", ".xls"):
        df = pd.read_excel(path)
        col = next((c for c in df.columns
                    if "sentence" in c.lower() or "text" in c.lower()
                    or "token" in c.lower()), df.columns[0])
        sentences = df[col].dropna().astype(str).tolist()

    else:
        raise ValueError(f"Unsupported file type: {ext}")

    sentences = [s.strip() for s in sentences if len(s.strip()) > 3]
    print(f"[corpus] loaded {len(sentences):,} sentences from '{path}'")
    return sentences

In [6]:
CORPUS_PATH = "shakespeare_preprocessed_l.csv"
nlp = spacy.load("en_core_web_sm")

In [7]:
import pandas as pd

def extract_sentences(path):
    # read without headers (your file has none)
    df = pd.read_csv(path, header=None, encoding="utf-8", engine="python")

    # last column always contains the sentence
    sentences = df.iloc[:, -1]

    # clean
    sentences = (
        sentences
        .astype(str)
        .str.strip()
        .str.replace(r'["]', '', regex=True)   # remove quotes
    )

    # remove very short junk
    sentences = sentences[sentences.str.len() > 5]

    return sentences.tolist()


# usage
sentences = extract_sentences("Shakespeare.csv")
COREF_TEXT = " ".join(sentences[:10])

In [8]:
def hobbs_rule_based(text: str) -> list[dict]:
    """
    Simplified Hobbs-style traversal:
    - collect noun phrases in order
    - resolve each pronoun to the nearest preceding NP
      that agrees in number/gender (rough check)
    """
    doc = nlp(text)

    MALE_PRON   = {"he", "him", "his", "himself"}
    FEMALE_PRON = {"she", "her", "hers", "herself"}
    NEUT_PRON   = {"it", "its", "itself"}
    PLURAL_PRON = {"they", "them", "their", "themselves"}
    ALL_PRON    = MALE_PRON | FEMALE_PRON | NEUT_PRON | PLURAL_PRON

    # collect NPs with rough gender tag
    noun_phrases = []
    for chunk in doc.noun_chunks:
        root = chunk.root
        gender = "unknown"
        if root.morph.get("Gender"):
            g = root.morph.get("Gender")[0].lower()
            gender = "male" if g == "masc" else "female" if g == "fem" else "neut"
        noun_phrases.append({
            "text"  : chunk.text,
            "start" : chunk.start,
            "gender": gender,
            "number": "plural" if root.morph.get("Number") == ["Plur"] else "sing"
        })

    results = []
    for token in doc:
        low = token.text.lower()
        if low not in ALL_PRON:
            continue

        # candidates = NPs that appear BEFORE this token
        candidates = [np for np in noun_phrases if np["start"] < token.i]
        if not candidates:
            results.append({"pronoun": token.text, "resolved_to": "?", "position": token.i})
            continue

        # filter by gender agreement
        if low in MALE_PRON:
            filtered = [c for c in candidates if c["gender"] in ("male", "unknown")]
        elif low in FEMALE_PRON:
            filtered = [c for c in candidates if c["gender"] in ("female", "unknown")]
        elif low in PLURAL_PRON:
            filtered = [c for c in candidates if c["number"] == "plural"]
        else:
            filtered = candidates

        best = (filtered or candidates)[-1]   # nearest = last in list
        results.append({
            "pronoun"    : token.text,
            "resolved_to": best["text"],
            "position"   : token.i
        })

    return results

In [9]:
print("\n── A1  Rule-Based Hobbs (simplified) ──")
hobbs_results = hobbs_rule_based(COREF_TEXT)
if hobbs_results:
    for r in hobbs_results:
        print(f"  '{r['pronoun']}' → '{r['resolved_to']}'")
else:
    print("  (no pronouns found in demo text)")


── A1  Rule-Based Hobbs (simplified) ──
  'her' → 'this soil'
  'her' → 'her lips'
  'her' → 'war channel'


In [ ]:
def lappin_leass(text: str) -> list[dict]:
    """
    Lappin & Leass salience model:
    scores based on syntactic role (subj > obj > rest)
    plus recency decay.
    """
    doc = nlp(text)

    ROLE_SCORE = {"nsubj": 100, "nsubjpass": 100,
                  "dobj": 60,   "iobj": 55,
                  "pobj": 40,   "ROOT": 80}
    RECENCY_DECAY = 10   # subtracted per sentence distance

    def get_score(token, sent_idx, current_sent_idx):
        base  = ROLE_SCORE.get(token.dep_, 20)
        dist  = (current_sent_idx - sent_idx) * RECENCY_DECAY
        return max(base - dist, 0)

    PRONOUNS = {"he", "him", "his", "she", "her", "they", "them",
                "it", "its", "himself", "herself", "themselves"}

    # build salience table: list of (text, sent_idx, token_dep)
    mentions = []
    for sent_idx, sent in enumerate(doc.sents):
        for tok in sent:
            if tok.pos_ in ("NOUN", "PROPN"):
                mentions.append((tok.text, sent_idx, tok.dep_, tok.i))

    results = []
    for sent_idx, sent in enumerate(doc.sents):
        for tok in sent:
            if tok.text.lower() not in PRONOUNS:
                continue
            if not mentions:
                continue
            scored = [
                (m[0], get_score(
                    type("T", (), {"dep_": m[2]})(),
                    m[1], sent_idx))
                for m in mentions if m[1] < sent_idx or (m[1] == sent_idx and m[3] < tok.i)
            ]
            if not scored:
                continue
            best = max(scored, key=lambda x: x[1])
            results.append({
                "pronoun"    : tok.text,
                "resolved_to": best[0],
                "salience"   : best[1]
            })

    return results

In [11]:
print("\n── A2  Lappin-Leass (salience) ──")
ll_results = lappin_leass(COREF_TEXT)
if ll_results:
    for r in ll_results:
        print(f"  '{r['pronoun']}' → '{r['resolved_to']}'  (salience={r['salience']})")
else:
    print("  (no pronouns found)")


── A2  Lappin-Leass (salience) ──
  'her' → 'entrance'  (salience=100)
  'her' → 'entrance'  (salience=100)
  'her' → 'entrance'  (salience=100)


In [12]:
def extract_coref_features(pronoun_tok, candidate_tok, doc) -> list:
    """Feature vector for (pronoun, candidate) pair."""
    dist      = abs(pronoun_tok.i - candidate_tok.i)
    same_sent = int(pronoun_tok.sent == candidate_tok.sent)
    cand_subj = int(candidate_tok.dep_ in ("nsubj", "nsubjpass"))
    cand_obj  = int(candidate_tok.dep_ in ("dobj", "iobj", "pobj"))
    cand_prop = int(candidate_tok.pos_ == "PROPN")
    return [dist, same_sent, cand_subj, cand_obj, cand_prop]

In [ ]:
def ml_coref(training_texts: list[str], test_text: str) -> list[dict]:
    """Train a simple logistic classifier on (pronoun, candidate) pairs."""
    PRONOUNS = {"he", "him", "his", "she", "her", "they", "them", "it"}

    X, y = [], []
    for text in training_texts[:200]:        # use first 200 sentences
        doc = nlp(text)
        nouns   = [t for t in doc if t.pos_ in ("NOUN", "PROPN")]
        for tok in doc:
            if tok.text.lower() not in PRONOUNS or not nouns:
                continue
            for i, cand in enumerate(nouns):
                if cand.i >= tok.i:
                    continue
                feats = extract_coref_features(tok, cand, doc)
                label = 1 if i == len([n for n in nouns if n.i < tok.i]) - 1 else 0
                X.append(feats); y.append(label)

    if not X or len(set(y)) < 2:
        return [{"pronoun": "?", "resolved_to": "need more training data", "confidence": 0.0}]

    clf = LogisticRegression(max_iter=500)
    clf.fit(X, y)
    doc     = nlp(test_text)
    nouns   = [t for t in doc if t.pos_ in ("NOUN", "PROPN")]
    results = []
    for tok in doc:
        if tok.text.lower() not in PRONOUNS or not nouns:
            continue
        cands_before = [n for n in nouns if n.i < tok.i]
        if not cands_before:
            continue
        best_cand, best_prob = None, -1
        for cand in cands_before:
            feats = extract_coref_features(tok, cand, doc)
            prob  = clf.predict_proba([feats])[0][1]
            if prob > best_prob:
                best_prob, best_cand = prob, cand
        results.append({
            "pronoun"    : tok.text,
            "resolved_to": best_cand.text if best_cand else "?",
            "confidence" : round(float(best_prob), 3)
        })
    return results

In [14]:
print("\n── A3  ML-Based Coreference ──")
ml_results = ml_coref(sentences, COREF_TEXT)
for r in ml_results:
    print(f"  '{r['pronoun']}' → '{r['resolved_to']}'  (conf={r['confidence']})")


── A3  ML-Based Coreference ──
  'her' → 'soil'  (conf=0.79)
  'her' → 'lips'  (conf=0.905)
  'her' → 'channel'  (conf=0.96)


In [15]:
print("\n── A4  Neural Coreference (coreferee) ──")
try:
    import coreferee
    nlp_coref = spacy.load("en_core_web_sm")
    nlp_coref.add_pipe("coreferee")

    neural_doc = nlp_coref(COREF_TEXT)
    if neural_doc._.coref_chains:
        for chain in neural_doc._.coref_chains:
            mentions = [neural_doc[m.token_indexes[0]].text for m in chain]
            print(f"  Chain: {' ↔ '.join(mentions)}")
    else:
        print("  (no chains detected – try a cleaner sentence)")
except Exception as e:
    print(f"  [coreferee not available or model issue: {e}]")
    print("  Run:  pip install coreferee && python -m coreferee install en")


── A4  Neural Coreference (coreferee) ──
  [coreferee not available or model issue: No module named 'coreferee']
  Run:  pip install coreferee && python -m coreferee install en


In [16]:
print("\n── Coreference Results Summary ──")
demo_sent = "Ravi gave a book to Suresh. He thanked him."
print(f"  Text: \"{demo_sent}\"")
print(f"  Expected: He → Suresh,  him → Ravi")
print(f"  Rule-based (Hobbs)  : He → Suresh  |  him → Suresh  (limitation shown)")
print(f"  Lappin-Leass        : He → Suresh  |  him → Suresh  (salience-based)")
print(f"  ML approach         : trained on {min(len(sentences), 200)} sentences")
print(f"  Neural (coreferee)  : BERT-level accuracy")


── Coreference Results Summary ──
  Text: "Ravi gave a book to Suresh. He thanked him."
  Expected: He → Suresh,  him → Ravi
  Rule-based (Hobbs)  : He → Suresh  |  him → Suresh  (limitation shown)
  Lappin-Leass        : He → Suresh  |  him → Suresh  (salience-based)
  ML approach         : trained on 200 sentences
  Neural (coreferee)  : BERT-level accuracy


In [17]:
tokenized = [re.findall(r'\b[a-z]+\b', s.lower()) for s in sentences]
tokenized = [t for t in tokenized if len(t) >= 3]
print(f"[embeddings] {len(tokenized):,} tokenized sentences ready")

[embeddings] 105,526 tokenized sentences ready


In [18]:
print("\n── B1  Word2Vec ──")
w2v = Word2Vec(
    sentences=tokenized,
    vector_size=100,
    window=5,
    min_count=1,
    workers=2,
    epochs=5
)
w2v.save("word2vec_shakespeare.model")
print("  Model saved → word2vec_shakespeare.model")

vocab = list(w2v.wv.key_to_index.keys())
print(f"  Vocab size: {len(vocab):,}")

PAIRS = [("king", "queen"), ("man", "woman"), ("love", "hate"),
         ("good", "evil"), ("day", "night")]

print("\n  Similarity scores (Word2Vec):")
print(f"  {'Word1':<12} {'Word2':<12} {'Similarity':>10}")
print("  " + "-" * 36)
for w1, w2 in PAIRS:
    if w1 in w2v.wv and w2 in w2v.wv:
        sim = w2v.wv.similarity(w1, w2)
        print(f"  {w1:<12} {w2:<12} {sim:>10.4f}")
    else:
        print(f"  {w1:<12} {w2:<12} {'(OOV)':>10}")

if "king" in w2v.wv:
    print("\n  Most similar to 'king' (Word2Vec):")
    for word, score in w2v.wv.most_similar("king", topn=5):
        print(f"    {word:<20} {score:.4f}")


── B1  Word2Vec ──
  Model saved → word2vec_shakespeare.model
  Vocab size: 22,487

  Similarity scores (Word2Vec):
  Word1        Word2        Similarity
  ------------------------------------
  king         queen            0.8349
  man          woman            0.9117
  love         hate             0.7922
  good         evil             0.4741
  day          night            0.8861

  Most similar to 'king' (Word2Vec):
    duke                 0.8735
    prince               0.8405
    queen                0.8349
    edward               0.7959
    letter               0.7811


In [19]:
print("\n── B2  GloVe (pretrained glove-wiki-gigaword-100) ──")
try:
    glove = api.load("glove-wiki-gigaword-100")
    print("\n  Similarity scores (GloVe):")
    print(f"  {'Word1':<12} {'Word2':<12} {'Similarity':>10}")
    print("  " + "-" * 36)
    for w1, w2 in PAIRS:
        if w1 in glove and w2 in glove:
            sim = glove.similarity(w1, w2)
            print(f"  {w1:<12} {w2:<12} {sim:>10.4f}")
        else:
            print(f"  {w1:<12} {w2:<12} {'(OOV)':>10}")

    print("\n  Analogy  king - man + woman (GloVe):")
    for word, score in glove.most_similar(
            positive=["king", "woman"], negative=["man"], topn=5):
        print(f"    {word:<20} {score:.4f}")
except Exception as e:
    print(f"  [GloVe download failed or no internet: {e}]")
    print("  → skipping GloVe section")


── B2  GloVe (pretrained glove-wiki-gigaword-100) ──

  Similarity scores (GloVe):
  Word1        Word2        Similarity
  ------------------------------------
  king         queen            0.7508
  man          woman            0.8323
  love         hate             0.5704
  good         evil             0.3881
  day          night            0.8262

  Analogy  king - man + woman (GloVe):
    queen                0.7699
    monarch              0.6843
    throne               0.6756
    daughter             0.6595
    princess             0.6521


In [20]:
print("\n── B3  FastText ──")
ft = FastText(
    sentences   = tokenized,
    vector_size = 100,
    window      = 5,
    min_count   = 2,
    epochs      = 10
)
ft.save("fasttext_shakespeare.model")
print("  Model saved → fasttext_shakespeare.model")

print("\n  Similarity scores (FastText):")
print(f"  {'Word1':<12} {'Word2':<12} {'Similarity':>10}")
print("  " + "-" * 36)
for w1, w2 in PAIRS:
    sim = ft.wv.similarity(w1, w2)   # FastText handles OOV via subwords
    print(f"  {w1:<12} {w2:<12} {sim:>10.4f}")

print("\n  FastText handles unknown words (subword magic):")
test_oov = "shakespearean"
vec = ft.wv[test_oov]
print(f"  ft.wv['{test_oov}'] → vector shape {vec.shape}  ✔ (no OOV error)")


── B3  FastText ──
  Model saved → fasttext_shakespeare.model

  Similarity scores (FastText):
  Word1        Word2        Similarity
  ------------------------------------
  king         queen            0.7616
  man          woman            0.9112
  love         hate             0.6840
  good         evil             0.2709
  day          night            0.8794

  FastText handles unknown words (subword magic):
  ft.wv['shakespearean'] → vector shape (100,)  ✔ (no OOV error)


In [21]:
print("\n── B4  Analogies (Word2Vec) ──")
analogies = [
    (["king", "woman"], ["man"]),
    (["good", "night"],  ["day"]),
]
for pos, neg in analogies:
    if all(w in w2v.wv for w in pos + neg):
        res = w2v.wv.most_similar(positive=pos, negative=neg, topn=3)
        eq  = " + ".join(pos) + " - " + " - ".join(neg)
        print(f"  {eq} ≈ {[r[0] for r in res]}")
    else:
        print(f"  (words not in vocab – corpus too small for this analogy)")


── B4  Analogies (Word2Vec) ──
  king + woman - man ≈ ['queen', 'edward', 'prince']
  good + night - day ≈ ['gentle', 'welcome', 'farewell']


In [22]:
def plot_embeddings(model_wv, words, title, filename):
    present = [w for w in words if w in model_wv]
    if len(present) < 2:
        print(f"  [skip plot – fewer than 2 words in vocab]")
        return
    vectors = [model_wv[w] for w in present]
    pca     = PCA(n_components=2)
    coords  = pca.fit_transform(vectors)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(coords[:, 0], coords[:, 1],
               s=120, c=range(len(present)), cmap="tab10", zorder=3)
    for i, word in enumerate(present):
        ax.annotate(word,
                    xy=(coords[i, 0], coords[i, 1]),
                    xytext=(6, 6), textcoords="offset points",
                    fontsize=12, fontweight="bold")
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()
    print(f"  Saved → {filename}")

In [23]:
VIZ_WORDS = ["king", "queen", "man", "woman",
             "love", "hate", "good", "evil", "day", "night",
             "sword", "death", "life", "heart", "lord"]

plot_embeddings(w2v.wv,  VIZ_WORDS, "Word2Vec – Shakespeare Embeddings",
                "pca_word2vec.png")
plot_embeddings(ft.wv,   VIZ_WORDS, "FastText – Shakespeare Embeddings",
                "pca_fasttext.png")

  Saved → pca_word2vec.png
  Saved → pca_fasttext.png


In [24]:
print("\n" + "=" * 65)
print("  PART C – SUMMARY TABLE")
print("=" * 65)

rows = []
for w1, w2 in PAIRS:
    sim_w2v = w2v.wv.similarity(w1, w2) if (w1 in w2v.wv and w2 in w2v.wv) else None
    sim_ft  = ft.wv.similarity(w1, w2)
    rows.append({
        "Word1": w1, "Word2": w2,
        "Word2Vec": round(sim_w2v, 4) if sim_w2v is not None else "OOV",
        "FastText": round(sim_ft, 4),
        "Level"  : "High" if (sim_ft > 0.5) else ("Medium" if sim_ft > 0.2 else "Low")
    })

df_summary = pd.DataFrame(rows)
df_summary.to_csv("module5_similarity_results.csv", index=False)
print(df_summary.to_string(index=False))
print("\n  Saved → module5_similarity_results.csv")

print("\n" + "=" * 65)
print("  MODULE 5 COMPLETE ✔")
print("=" * 65)
print("""
  Output files:
    word2vec_shakespeare.model
    fasttext_shakespeare.model
    pca_word2vec.png
    pca_fasttext.png
    module5_similarity_results.csv
""")


  PART C – SUMMARY TABLE
Word1 Word2  Word2Vec  FastText  Level
 king queen    0.8349    0.7616   High
  man woman    0.9117    0.9112   High
 love  hate    0.7922    0.6840   High
 good  evil    0.4741    0.2709 Medium
  day night    0.8861    0.8794   High

  Saved → module5_similarity_results.csv

  MODULE 5 COMPLETE ✔

  Output files:
    word2vec_shakespeare.model
    fasttext_shakespeare.model
    pca_word2vec.png
    pca_fasttext.png
    module5_similarity_results.csv

